In [5]:
from pathlib import Path
import pandas as pd
from sklearn.metrics import root_mean_squared_error
import ast
import numpy as np
import pandas as pd
from shapely.geometry import box
import geopandas as gpd
import matplotlib.pyplot as pyplot
from matplotlib.lines import Line2D

# Collect all data in a dictionary

In [9]:
folder = Path("csv/SCFG015x015_degree/")

lac = pd.DataFrame()
gac = pd.DataFrame()

for file in folder.rglob("*.csv"):
    df = pd.read_csv(file)
    lac = pd.concat([lac, df[df["datatype"] == "lac"]])
    gac = pd.concat([gac, df[df["datatype"] == "gac"]])

all_regions = set(lac["region"]).union(gac["region"])

regions = {
    region: {
        "lac": lac[lac["region"] == region],
        "gac": gac[gac["region"] == region]
    }
    for region in all_regions
}

print(lac.head())
print(gac.head())

# Example: access one region
# print(regions["San_Rafael_South_America"]["lac"])
# print(regions["San_Rafael_South_America"]["gac"])

        date datatype satellite      region  \
91  19920401      lac    noaa11  chl_Rafael   
93  19920402      lac    noaa11  chl_Rafael   
95  19920403      lac    noaa11  chl_Rafael   
97  19920404      lac    noaa11  chl_Rafael   
99  19920405      lac    noaa11  chl_Rafael   

                                                 data  
91  [nan, nan, nan, nan, nan, nan, nan, nan, nan, ...  
93  [nan, nan, nan, nan, nan, nan, nan, nan, nan, ...  
95  [nan, nan, nan, nan, nan, nan, nan, nan, nan, ...  
97  [nan, nan, nan, nan, nan, nan, nan, nan, nan, ...  
99  [nan, nan, nan, nan, nan, nan, nan, nan, nan, ...  
       date datatype satellite      region   data
0  19920101      gac    noaa11  chl_Rafael  [nan]
1  19920102      gac    noaa11  chl_Rafael  [0.0]
2  19920103      gac    noaa11  chl_Rafael  [nan]
3  19920104      gac    noaa11  chl_Rafael  [nan]
4  19920105      gac    noaa11  chl_Rafael  [nan]


# Calculate mean, max, min, sd, rmse

In [10]:
summary_rows = []

for data_df in [lac, gac]:

    for (region, satellite, datatype), group in data_df.groupby(
        ["region", "satellite", "datatype"]
    ):

        values = []

        for data_string in group["data"]:

            nums = [
                np.nan if x.strip() == "nan" else float(x)
                for x in data_string.strip("[]").split(",")
                if x.strip() != ""
            ]

            values.extend(nums)

        values = np.array(values, dtype=float)

        # remove NaNs
        values = values[~np.isnan(values)]

        summary_rows.append({
            "region": region,
            "satellite": satellite,
            "datatype": datatype,
            "mean": np.mean(values) if len(values) else np.nan,
            "std": np.std(values) if len(values) else np.nan,
            "count": len(values)
        })


summary = (
    pd.DataFrame(summary_rows)
    .sort_values(
        ["region", "satellite", "datatype"]
    )
    .reset_index(drop=True)
)



# RMSE calculation
# Compare only dates where both LAC and GAC have values


rmse_rows = []


for (region, satellite), lac_group in lac.groupby(
    ["region", "satellite"]
):

    # matching GAC
    gac_group = gac[
        (gac["region"] == region) &
        (gac["satellite"] == satellite)
    ]


    # group by date because there are multiple rows per day
    lac_dates = lac_group.groupby("date")
    gac_dates = gac_group.groupby("date")


    daily_rmse = []


    # only use dates existing in both datasets
    common_dates = set(lac_dates.groups.keys()).intersection(
        set(gac_dates.groups.keys())
    )


    for date in sorted(common_dates):

        lac_day = lac_dates.get_group(date)
        gac_day = gac_dates.get_group(date)


        # collect all LAC values for this date
        lac_values = []

        for s in lac_day["data"]:

            vals = [
                np.nan if x.strip() == "nan" else float(x)
                for x in s.strip("[]").split(",")
                if x.strip() != ""
            ]

            lac_values.extend(vals)


        lac_values = np.array(
            lac_values,
            dtype=float
        )

        lac_values = lac_values[
            ~np.isnan(lac_values)
        ]


        # collect GAC values for this date
        gac_values = []

        for s in gac_day["data"]:

            vals = [
                np.nan if x.strip() == "nan" else float(x)
                for x in s.strip("[]").split(",")
                if x.strip() != ""
            ]

            gac_values.extend(vals)


        gac_values = np.array(
            gac_values,
            dtype=float
        )

        gac_values = gac_values[
            ~np.isnan(gac_values)
        ]


        # skip date if either has no values
        if len(lac_values) == 0 or len(gac_values) == 0:
            continue


        # GAC is one value per date
        gac_value = gac_values[0]


        rmse = root_mean_squared_error(
            np.full(len(lac_values), gac_value),
            lac_values
        )


        daily_rmse.append(rmse)



    rmse_rows.append({

        "region": region,
        "satellite": satellite,

        # mean RMSE over all valid dates
        "rmse": np.mean(daily_rmse)
            if len(daily_rmse) > 0
            else np.nan,

        "days_used": len(daily_rmse)

    })



rmse_summary = (
    pd.DataFrame(rmse_rows)
    .sort_values(
        ["region", "satellite"]
    )
    .reset_index(drop=True)
)




# Display


print("Statistics:")
print(summary)


print("\nRMSE:")
print(rmse_summary)




# Save CSV files


summary.to_csv(
    "results/summary_statistics_SCFG015x015_degree.csv",
    index=False
)


rmse_summary.to_csv(
    "results/rmse_summary_SCFG015x015_degree.csv",
    index=False
)


print("\nSaved:")
print("results/summary_statistics_SCFG015x015_degree.csv")
print("results/rmse_summary_SCFG015x015_degree.csv")

Statistics:
           region satellite datatype       mean        std  count
0   can_Auyuittuq    noaa11      gac  91.837272  21.889493    674
1   can_Auyuittuq    noaa11      lac  84.177905  28.279585   1394
2   can_Auyuittuq    noaa14      gac  90.947689  21.348285    648
3   can_Auyuittuq    noaa14      lac  87.078903  25.190380   1678
4      chl_Rafael    noaa11      gac  14.217536  27.071683    460
5      chl_Rafael    noaa11      lac  32.358974  36.074766    416
6      chl_Rafael    noaa14      gac  13.145341  22.850461    508
7      chl_Rafael    noaa14      lac  38.040523  38.076962    510
8   nor_Jotunheim    noaa11      gac  74.403226  37.642033    310
9   nor_Jotunheim    noaa11      lac  60.297491  43.477111    558
10  nor_Jotunheim    noaa14      gac  77.135638  35.560073    376
11  nor_Jotunheim    noaa14      lac  75.154303  36.168579    674
12    rus_Karelia    noaa11      gac  30.839348  44.956694    532
13    rus_Karelia    noaa11      lac  22.471890  39.750699   100